# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Shile/flyrank_ml_internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

The task is ultimately a clustering task. Fot the diagnostic tool, it uses unsupervised machine learning in grouping low performance pages based on similar features which can be easily used in examining the possible cause of failure.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There is no real target here. Content-related data is used along with an engineered feature called perfomance score for the clustering. All performance metrics are eliminated from this final layer to prevent leakage because the performance score is like an aggregate of the these scores.

In [83]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report


In [92]:
df = pd.read_csv('/content/low_performers (2).csv')
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ai_sessions_90d_rank,scroll_events_90d_rank,days_with_impressions_rank,days_with_sessions_rank,ctr_rank,engagement_rate_rank,scroll_rate_rank,ai_traffic_pct_rank,avg_position_rank,performance_score
0,content_9a34b442b552,client_8722616204,0.0,0.00,LOW,0.00,keyword article,informational,3059.0,20810.0,...,0.469703,0.182409,0.129373,0.078523,0.211457,0.356447,0.183740,0.469703,0.746253,27.711781
1,content_a5a2fbc76336,client_8527a891e2,10.0,0.00,LOW,0.00,keyword article,informational,1342.0,8469.0,...,0.469703,0.489193,0.371314,0.322714,0.211457,0.356447,0.753838,0.469703,0.088478,36.914141
2,content_91067a14431a,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,2802.0,18013.0,...,0.469703,0.489193,0.334708,0.574088,0.211457,0.356447,0.428077,0.469703,0.199251,39.450592
3,content_9d548144b06d,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3455.0,21571.0,...,0.469703,0.182409,0.238131,0.574088,0.211457,0.356447,0.183740,0.469703,0.481136,35.380067
4,content_19ad8f9bac29,client_3fdba35f04,480.0,0.58,MEDIUM,1.97,keyword article,informational,1372.0,9052.0,...,0.469703,0.870940,0.263618,0.322714,0.211457,0.356447,0.907375,0.469703,0.025915,41.379357


In [ ]:
df['perfomance_label'] = df['performance_score'].apply(lambda x:  'not good' if x <= low_cut_off else 'good')

In [85]:
df.iloc[0]

,0
content_id,content_9a34b442b552
client_id,client_8722616204
search_volume,0.0
competition,0.0
competition_level,LOW
cpc,0.0
content_type,keyword article
main_intent,informational
word_count,3059.0
char_count,20810.0


In [86]:
performance_cols = ['impressions_90d', 'clicks_90d', 'pageviews_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'ctr', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'trend_pct', 'avg_position', 'performance_score', 'trend_direction']

for col in df.columns:
  if '_rank' in col:
    performance_cols.append(col)

df = df.drop(columns = performance_cols)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5171 entries, 0 to 5170
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              5171 non-null   object 
 1   client_id               5171 non-null   object 
 2   search_volume           5130 non-null   float64
 3   competition             5130 non-null   float64
 4   competition_level       5130 non-null   object 
 5   cpc                     5130 non-null   float64
 6   content_type            5171 non-null   object 
 7   main_intent             5171 non-null   object 
 8   word_count              3624 non-null   float64
 9   char_count              3624 non-null   float64
 10  provider_used           1549 non-null   object 
 11  model_used              4254 non-null   object 
 12  content_age_days        5171 non-null   int64  
 13  days_since_last_update  5171 non-null   int64  
dtypes: float64(5), int64(2), object(7)
memor

In [87]:
df = df.drop(columns = ['main_intent'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5171 entries, 0 to 5170
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              5171 non-null   object 
 1   client_id               5171 non-null   object 
 2   search_volume           5130 non-null   float64
 3   competition             5130 non-null   float64
 4   competition_level       5130 non-null   object 
 5   cpc                     5130 non-null   float64
 6   content_type            5171 non-null   object 
 7   word_count              3624 non-null   float64
 8   char_count              3624 non-null   float64
 9   provider_used           1549 non-null   object 
 10  model_used              4254 non-null   object 
 11  content_age_days        5171 non-null   int64  
 12  days_since_last_update  5171 non-null   int64  
dtypes: float64(5), int64(2), object(6)
memory usage: 525.3+ KB


In [88]:
df.rename(columns = {'perfomance_label':'performance_label'}, inplace = True)
df

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,word_count,char_count,provider_used,model_used,content_age_days,days_since_last_update
0,content_9a34b442b552,client_8722616204,0.0,0.00,LOW,0.00,keyword article,3059.0,20810.0,google,gemini-3-flash-preview,90,20
1,content_a5a2fbc76336,client_8527a891e2,10.0,0.00,LOW,0.00,keyword article,1342.0,8469.0,NaN,gpt-4o-mini,238,103
2,content_9d548144b06d,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,3455.0,21571.0,google,gemini-3-flash-preview,118,20
3,content_249298388b45,client_2c624232cd,590.0,0.00,LOW,0.00,keyword article,NaN,NaN,NaN,gpt-4o-mini,438,22
4,content_4595e8704e07,client_8527a891e2,90.0,0.06,LOW,0.03,keyword article,3666.0,21824.0,NaN,gpt-4o-mini,348,104
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5166,content_b1d45033b059,client_e29c9c180c,0.0,0.00,LOW,0.00,keyword article,5328.0,35141.0,NaN,gemini-2.5-flash,225,20
5167,content_be106cd29636,client_d029fa3a95,0.0,0.00,LOW,0.00,comparison article,4220.0,29604.0,NaN,gemini-2.5-flash,225,20
5168,content_7ba9b154acf6,client_19581e27de,170.0,0.16,LOW,2.75,keyword article,NaN,NaN,NaN,NaN,482,22
5169,content_9bb9a0584cae,client_8527a891e2,210.0,0.00,LOW,0.00,keyword article,1446.0,9001.0,NaN,gpt-4o-mini,146,8


In [89]:
df.shape

(5171, 13)

In [90]:
df.columns

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'word_count', 'char_count',
       'provider_used', 'model_used', 'content_age_days',
       'days_since_last_update'],
      dtype='object')

In [91]:
X = df.drop(columns = ['performance_label']).copy()
y = df['performance_label'].copy()

KeyError: "['performance_label'] not found in axis"

In [ ]:
cat_cols = ['content_id', 'client_id', 'competition_level', 'content_type', 'provider_used', 'model_used']
num_cols = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update']

len(cat_cols) + len(num_cols)


In [ ]:
for i in cat_cols:
  print(f'{i}: {X[i].nunique()}')
  print()

In [ ]:
X.info()

In [ ]:
for j in num_cols:
  print(f'{j}: {X[j].describe()}')
  print()

In [ ]:
X['search_volume'] = np.log1p(X['search_volume'])
X['cpc'] = np.log1p(X['cpc'])

In [ ]:
X[num_cols].corr()

In [ ]:
X_features = X.drop(columns = ['word_count'], axis = 1).copy()

In [ ]:
client_group = X['client_id']
content_group = X['content_id']

In [ ]:
X_features = X_features.drop(columns = ['client_id', 'content_id'], axis = 1)

In [ ]:
X_features

In [ ]:
cat_cols = ['competition_level', 'content_type', 'provider_used', 'model_used']
num_cols = ['search_volume', 'competition', 'cpc', 'char_count', 'content_age_days', 'days_since_last_update']

len(cat_cols) + len(num_cols)


In [ ]:
num_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
cat_pipeline = Pipeline(
    steps = [
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num_pipeline', num_pipeline, num_cols),
        ('cat_pipeline', cat_pipeline, cat_cols)
    ])

pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression())
    ]
)

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_features, y, groups = client_group))

X_train, X_test = X_features.iloc[train_idx].copy(), X_features.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

In [ ]:
X_train.info()

In [ ]:
X_test.info()

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
pipeline.predict_proba(X_test)

In [ ]:
pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)
prediction_probabilities = pipeline.predict_proba(X_test)
print()
print(classification_report(y_test, predictions))

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.